# Práctica 10: Detección de Objetos con Deep Learning

Vas a reutilizar el pipeline de `cv2.dnn` con YOLOv4-tiny visto en `../notebooks/10_deteccion_objetos_dnn.ipynb`, pero sobre una imagen distinta y contando cuántas detecciones hay de una clase específica. También vas a comparar qué pasa al cambiar el umbral de confianza. Si te trabás, repasá la lección.

Recordá haber corrido `python scripts/descargar_modelos.py` desde la raíz del repo antes de empezar, para tener los archivos del modelo en `../modelos/`.

### Ejercicio 1: Contar detecciones de una clase específica
La lección usó `../imgs/img1.jpg`. Ahora corré el detector sobre `../imgs/img2.jpg` y contá cuántas detecciones hay de una clase que vos elijas (por ejemplo `"person"` o `"car"`), imprimiendo el conteo final.

In [ ]:
import cv2  # OpenCV, incluye el submódulo cv2.dnn para cargar y ejecutar la red
import numpy as np  # NumPy para manejar los arrays de detecciones/cajas

net = cv2.dnn.readNetFromDarknet('../modelos/yolov4-tiny.cfg', '../modelos/yolov4-tiny.weights')  # Carga la arquitectura (.cfg) y los pesos entrenados (.weights) de YOLOv4-tiny en formato Darknet, igual que en la lección

with open('../modelos/coco.names', 'r') as archivo:
    clases = [linea.strip() for linea in archivo.readlines()]  # Lee los nombres de las 80 clases de COCO, un nombre por línea; el índice de la lista coincide con el ID de clase que devuelve la red

output_layers = net.getUnconnectedOutLayersNames()  # Nombres de las capas de salida de la red, necesarios para leer las detecciones con net.forward()

imagen = cv2.imread('../imgs/img2.jpg')  # Esta práctica usa img2.jpg en vez de img1.jpg (la de la lección)
alto, ancho = imagen.shape[:2]  # Dimensiones reales de la imagen, para desnormalizar las coordenadas que devuelve la red

blob = cv2.dnn.blobFromImage(imagen, scalefactor=1/255, size=(416, 416), swapRB=True, crop=False)  # Convierte la imagen en un blob apto para la red: redimensiona a 416x416 (tamaño de entrada de YOLOv4-tiny), normaliza a [0,1] dividiendo por 255, y convierte BGR->RGB con swapRB porque la red fue entrenada con imágenes en RGB
net.setInput(blob)  # Carga el blob como entrada de la red
detecciones = net.forward(output_layers)  # Ejecuta la inferencia y devuelve las detecciones crudas de cada capa de salida

CONFIANZA_MINIMA = 0.5  # Umbral de confianza, mismo valor que en la lección para poder comparar resultados entre imágenes

cajas = []
confianzas = []
ids_clases = []

for salida in detecciones:  # Recorre las detecciones de cada capa de salida (YOLO tiene varias escalas)
    for deteccion in salida:  # Recorre cada detección individual (una por anchor box)
        puntajes = deteccion[5:]  # A partir del índice 5 vienen los puntajes de probabilidad de las 80 clases (los primeros 5 son caja + objectness)
        id_clase = np.argmax(puntajes)  # Clase con mayor puntaje
        confianza = puntajes[id_clase]  # Puntaje de esa clase más probable

        if confianza > CONFIANZA_MINIMA:  # Descarta detecciones poco confiables
            centro_x = int(deteccion[0] * ancho)  # Desnormaliza el centro X multiplicando por el ancho real de la imagen
            centro_y = int(deteccion[1] * alto)  # Desnormaliza el centro Y multiplicando por el alto real
            w = int(deteccion[2] * ancho)  # Ancho de la caja desnormalizado
            h = int(deteccion[3] * alto)  # Alto de la caja desnormalizado

            x = int(centro_x - w / 2)  # Convierte de (centro, tamaño) a esquina superior izquierda, formato que espera cv2.rectangle
            y = int(centro_y - h / 2)  # Ídem para Y

            cajas.append([x, y, w, h])
            confianzas.append(float(confianza))
            ids_clases.append(id_clase)

indices = cv2.dnn.NMSBoxes(cajas, confianzas, CONFIANZA_MINIMA, nms_threshold=0.4)  # Non-Max Suppression: elimina cajas duplicadas que apuntan al mismo objeto, quedándose con la de mayor confianza entre las que se solapan más de un 40% (IoU)

# TODO: definí la clase que querés contar (tiene que ser un nombre válido de ../modelos/coco.names)
# Qué: `clase_a_contar` es el nombre exacto de la clase (string) sobre el que vas a filtrar las detecciones finales.
clase_a_contar = ""

# TODO: recorré `indices`, fijate la clase de cada detección con `clases[ids_clases[i]]`
# y contá cuántas coinciden con `clase_a_contar`. Guardá el resultado en `conteo`.
# Qué: por cada índice sobreviviente de NMS hay que comparar el nombre de su clase contra `clase_a_contar` y acumular las coincidencias en `conteo`.
conteo = None

print(f"Detecciones de '{clase_a_contar}': {conteo}")

### Ejercicio 2: Comparar umbrales de confianza
Repetí la detección sobre la misma imagen (`../imgs/img2.jpg`) pero con un umbral de confianza más bajo (por ejemplo `0.3` en vez de `0.5`) y compará cuántas detecciones totales aparecen de más (o de menos) respecto del Ejercicio 1.

In [ ]:
# TODO: definí el nuevo umbral de confianza a probar (por ejemplo 0.3)
# Qué: `CONFIANZA_MINIMA_BAJA` reemplaza a CONFIANZA_MINIMA en el filtro de detecciones para ver cómo cambian los resultados con un umbral más permisivo.
CONFIANZA_MINIMA_BAJA = None

blob = cv2.dnn.blobFromImage(imagen, scalefactor=1/255, size=(416, 416), swapRB=True, crop=False)  # Vuelve a generar el blob de la misma imagen (mismo preprocesamiento: redimensiona a 416x416, normaliza a [0,1], BGR->RGB)
net.setInput(blob)
detecciones = net.forward(output_layers)  # Vuelve a correr la inferencia; las detecciones crudas son las mismas que antes, lo que cambia es el umbral de filtrado más abajo

cajas_bajo = []
confianzas_bajo = []
ids_clases_bajo = []

for salida in detecciones:
    for deteccion in salida:
        puntajes = deteccion[5:]  # Puntajes de las 80 clases de COCO para esta detección
        id_clase = np.argmax(puntajes)  # Clase con mayor puntaje
        confianza = puntajes[id_clase]  # Puntaje de esa clase

        # TODO: filtrá usando CONFIANZA_MINIMA_BAJA en vez de CONFIANZA_MINIMA
        # Qué: la condición tiene que comparar `confianza` contra el nuevo umbral más bajo, no contra CONFIANZA_MINIMA (que se usó en el ejercicio anterior).
        if confianza > CONFIANZA_MINIMA_BAJA:
            centro_x = int(deteccion[0] * ancho)  # Desnormaliza el centro X
            centro_y = int(deteccion[1] * alto)  # Desnormaliza el centro Y
            w = int(deteccion[2] * ancho)  # Ancho desnormalizado
            h = int(deteccion[3] * alto)  # Alto desnormalizado

            x = int(centro_x - w / 2)  # Esquina superior izquierda X
            y = int(centro_y - h / 2)  # Esquina superior izquierda Y

            cajas_bajo.append([x, y, w, h])
            confianzas_bajo.append(float(confianza))
            ids_clases_bajo.append(id_clase)

# TODO: aplicá cv2.dnn.NMSBoxes con CONFIANZA_MINIMA_BAJA como score_threshold
# Qué: hay que correr Non-Max Suppression sobre `cajas_bajo`/`confianzas_bajo` para eliminar duplicados, usando CONFIANZA_MINIMA_BAJA como umbral de score (mismo nms_threshold=0.4 que en la lección).
indices_bajo = None

print(f"Detecciones con umbral {CONFIANZA_MINIMA}: {len(indices)}")
print(f"Detecciones con umbral {CONFIANZA_MINIMA_BAJA}: {len(indices_bajo)}")

# TODO: calculá la diferencia (indices_bajo - indices) e imprimí una conclusión:
# ¿aparecieron más detecciones (probablemente con más falsos positivos) al bajar el umbral?
# Qué: comparar la cantidad de detecciones de ambos umbrales y reflexionar sobre el trade-off entre sensibilidad (detectar más objetos reales) y falsos positivos (aceptar detecciones poco confiables) al bajar CONFIANZA_MINIMA.